# Final Multiclass Labeling

Use this notebook only after selecting the final classifier checkpoint you want to trust for unlabeled pseudo-labeling.

Workflow:
- load the multiclass data config
- point to the final chosen checkpoint
- run unlabeled prediction with a confidence threshold
- save the resulting pseudo-label candidates to `experiments/classifier/multiclass/x64/training/artifacts/multiclass_classifier_50k/unlabeled_predictions.csv`


In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src" / "wafer_defect").exists() and (candidate / "configs").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root containing src/wafer_defect and configs/")

SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

NOTEBOOK_DIR = REPO_ROOT / "experiments/classifier/multiclass/x64/final_labeling"
TRAINING_DIR = NOTEBOOK_DIR.parent / "training"
TRAINING_ARTIFACT_DIR = TRAINING_DIR / "artifacts" / "multiclass_classifier_50k"


In [ ]:
data_config_path = TRAINING_DIR / "data_config.toml"
checkpoint_path = TRAINING_ARTIFACT_DIR / "best_model.pt"
confidence_threshold = 0.98

print(f"Data config: {data_config_path}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Confidence threshold: {confidence_threshold:.2f}")


In [ ]:
predict_command = [
    sys.executable,
    str(REPO_ROOT / "scripts/classifier/predict_unlabeled_multiclass.py"),
    "--config",
    str(data_config_path),
    "--checkpoint",
    str(checkpoint_path),
    "--min-confidence",
    str(confidence_threshold),
]
print("Running:", " ".join(predict_command))
subprocess.run(predict_command, cwd=REPO_ROOT, check=True)


In [ ]:
predictions_path = TRAINING_ARTIFACT_DIR / "unlabeled_predictions.csv"
predictions = pd.read_csv(predictions_path)

display(predictions.head())
display(predictions["accepted_for_pseudo_label"].value_counts().rename_axis("accepted").to_frame("count"))
print("Accepted fraction:", predictions["accepted_for_pseudo_label"].mean())
